In [ ]:
# CELL 1 — Drive and recovered experiment inputs (read-only)
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/thesis_DR')
if not ROOT.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
assert ROOT.is_dir(), f'Google Drive project folder not found: {ROOT}'

FILES = {
    11: ('gbls_exp11_morphology_features_v1.csv', 'gbls_exp11_results.csv'),
    12: ('sr_gbls_exp12_results.csv',),
    13: ('sr_gbls_exp13_fair_coverage.csv',),
    14: ('sr_gbls_exp14_idrid_grade2_audit.csv',),
    15: ('sr_gbls_5pct_oracle_feasibility.csv',),
}
for exp, names in FILES.items():
    for name in names:
        p = ROOT / name
        if not p.is_file():
            raise FileNotFoundError(f'Experiment {exp} source result missing: {p}')
        print(f'Exp {exp}: {name}')

feat11 = pd.read_csv(ROOT / FILES[11][0])
res11 = pd.read_csv(ROOT / FILES[11][1])
res12 = pd.read_csv(ROOT / FILES[12][0])
res13 = pd.read_csv(ROOT / FILES[13][0])
res14 = pd.read_csv(ROOT / FILES[14][0])
res15 = pd.read_csv(ROOT / FILES[15][0])
SEEDS = {42, 43, 44, 45, 46}

# These are the existing experiment outputs. No training or segmentation is rerun.
# The original execution code for Experiments 12–15 was not recovered.
# Subsequent cells reproduce analyses from the saved records, without modifying them.


In [ ]:
# CELL 2 — Experiment 11: GBLS morphology feature and result audit
expected = {'APTOS': 3504, 'IDRiD': 507, 'Messidor-2': 1740}
required = {'dataset', 'image_id'}
assert required.issubset(feat11.columns)
feature_cols = [c for c in feat11.columns if c not in ('dataset', 'image_id')]
assert len(feature_cols) == 32, f'Expected 32 morphology features, got {len(feature_cols)}'
assert not feat11.duplicated(['dataset', 'image_id']).any()
assert feat11.groupby('dataset').size().to_dict() == expected
assert np.isfinite(feat11[feature_cols].to_numpy(float)).all()

must = {'seed','dataset','analysis','method','n','auto_clear_n',
        'missed_referable_n','coverage_pct','risk_pct','grade2_miss_pct'}
assert must.issubset(res11.columns), must - set(res11.columns)
assert set(res11.seed) == SEEDS
assert not res11.duplicated(['seed','dataset','analysis','method']).any()

# Arithmetic verification of recorded risk and coverage
assert (res11.auto_clear_n <= res11.n).all()
assert (res11.missed_referable_n <= res11.auto_clear_n).all()
calc_cov = 100 * res11.auto_clear_n / res11.n
calc_risk = 100 * res11.missed_referable_n / res11.auto_clear_n.replace(0,np.nan)
assert np.allclose(calc_cov, res11.coverage_pct, atol=1e-5)
assert np.allclose(calc_risk, res11.risk_pct, atol=1e-5, equal_nan=True)

summary11 = (res11.loc[res11.analysis.eq('source_LTT')]
    .groupby(['dataset','method'])
    .agg(n_seeds=('seed','nunique'),
         coverage_mean=('coverage_pct','mean'), coverage_sd=('coverage_pct','std'),
         risk_mean=('risk_pct','mean'), risk_sd=('risk_pct','std'))
    .round(3))
assert summary11.n_seeds.eq(5).all()
print('Morphology features:', len(feature_cols), 'across', len(feat11), 'unique images')
print('\nRecorded Experiment 11 source-selected results\n',summary11.to_string())


In [ ]:
# CELL 3 — Experiment 12: SR-GBLS model-selection and outcome audit
required = {'seed','dataset','analysis','method','selected_k','C','boundary_weight',
            'selected_features','n','auto_clear_n','missed_referable_n',
            'coverage_pct','risk_pct','grade2_miss_pct'}
assert required.issubset(res12.columns), required - set(res12.columns)
assert len(res12) == 30, f'Expected 30 rows, got {len(res12)}'
assert set(res12.seed) == SEEDS
assert set(res12.method) == {'SR_GBLS'}
assert not res12.duplicated(['seed','dataset','analysis','method']).any()

for row in res12.itertuples(index=False):
    selected = [item.strip() for item in str(row.selected_features).split(';') if item.strip()]
    assert len(selected) == int(row.selected_k), (row.seed,row.dataset,'feature count mismatch')
    assert len(set(selected)) == len(selected)
    assert set(selected).issubset(feature_cols), (row.seed,set(selected)-set(feature_cols))

calc_cov = 100 * res12.auto_clear_n / res12.n
calc_risk = 100 * res12.missed_referable_n / res12.auto_clear_n.replace(0,np.nan)
assert np.allclose(calc_cov, res12.coverage_pct, atol=1e-5)
assert np.allclose(calc_risk, res12.risk_pct, atol=1e-5, equal_nan=True)

selection = (res12.loc[(res12.dataset=='APTOS') & (res12.analysis=='source_LTT'),
                       ['seed','selected_k','C','boundary_weight','selected_features']]
             .sort_values('seed'))
summary12 = (res12.loc[res12.analysis.eq('source_LTT')]
    .groupby('dataset')
    .agg(n_seeds=('seed','nunique'),
         coverage_mean=('coverage_pct','mean'),coverage_sd=('coverage_pct','std'),
         risk_mean=('risk_pct','mean'),risk_sd=('risk_pct','std'))
    .round(3))
assert summary12.n_seeds.eq(5).all()
print('Source-selected specialist parameters\n',
      selection[['seed','selected_k','C','boundary_weight']].to_string(index=False))
print('\nRecorded SR-GBLS source-selected results\n', summary12.to_string())


In [ ]:
# CELL 4 — Experiment 13: retrospective matched-coverage comparison and Fig. 5
import matplotlib.pyplot as plt

required = {'dataset','seed','method','target_coverage_pct','actual_coverage_pct',
            'auto_clear_n','missed_referable_n','risk_pct',
            'boundary_tie_split','cutoff_is_deployable'}
assert required.issubset(res13.columns), required-set(res13.columns)
assert len(res13) == 270
assert set(res13.seed) == SEEDS
assert not res13.duplicated(['dataset','seed','method','target_coverage_pct']).any()

as_bool = lambda series: series.astype(str).str.lower().eq('true')
assert not as_bool(res13.cutoff_is_deployable).any(), \
       'Unexpected deployable cutoff present: do not label this analysis retrospective without checking.'

calc_risk = 100 * res13.missed_referable_n / res13.auto_clear_n.replace(0,np.nan)
assert np.allclose(calc_risk, res13.risk_pct, atol=1e-5, equal_nan=True)

matched = (res13.groupby(['dataset','method','target_coverage_pct'])
    .agg(n_seeds=('seed','nunique'), coverage=('actual_coverage_pct','mean'),
         risk=('risk_pct','mean'), risk_sd=('risk_pct','std'))
    .reset_index())
assert matched.n_seeds.eq(5).all()
print('Recorded non-deployable matched-coverage points:',len(res13))
print('Boundary-tie splits:',int(as_bool(res13.boundary_tie_split).sum()))
print('\nMatched-coverage summary at 20%, 30%, 35%, 40%\n',
    matched[matched.target_coverage_pct.isin([20,30,35,40])].round(3).to_string(index=False))

methods = [('Lesion_only','Lesion-only'),
           ('GBLS_morphology','GBLS'), ('SR_GBLS','SR-GBLS')]
fig, axes = plt.subplots(1,2,figsize=(11.5,4.4),sharey=False)
for ax, dataset, panel in zip(axes, ['IDRiD','Messidor-2'], ['(a)','(b)']):
    for raw_name, label in methods:
        part = matched[(matched.dataset==dataset)&(matched.method==raw_name)]
        if part.empty:
            raise ValueError(f'Expected method {raw_name} not present in {dataset}')
        part = part.sort_values('target_coverage_pct')
        ax.errorbar(part.coverage,part.risk,yerr=part.risk_sd,marker='o',
                    linewidth=1.5,markersize=4,capsize=2,label=label)
    ax.axhline(5,linestyle='--',color='gray',linewidth=1,label='5% target')
    ax.set(title=f'{panel} {dataset}',xlabel='Auto-clear coverage (%)',
           ylabel='Observed auto-clear risk (%)')
    ax.grid(alpha=0.2)
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=8)
fig.tight_layout()
# Preview only. No Drive files are overwritten or created.
plt.show()
plt.close(fig)


In [ ]:
# CELL 5 — Experiment 14: IDRiD grade-2 failure audit
required = {'image_id','original_grade','binary_label','lesion_score','seed',
            'SR_score','SR_source_threshold','SR_auto_clear','lesion_only_auto_clear'}
assert required.issubset(res14.columns), required-set(res14.columns)
assert len(res14) == 507*5
assert set(res14.seed) == SEEDS
assert not res14.duplicated(['seed','image_id']).any()
assert (res14.groupby('seed').size() == 507).all()
assert np.array_equal((res14.original_grade>=2).astype(int),res14.binary_label.astype(int))

sr_clear = as_bool(res14.SR_auto_clear)
lesion_clear = as_bool(res14.lesion_only_auto_clear)
# Disposition consistency with the saved source-selected threshold
predicted_clear = res14.SR_score.le(res14.SR_source_threshold)
assert np.array_equal(sr_clear.to_numpy(), predicted_clear.to_numpy()), \
       'SR_auto_clear differs from the stored score/threshold comparison.'

report = []
for seed, d in res14.groupby('seed'):
    clear = as_bool(d.SR_auto_clear)
    g2 = d.original_grade.eq(2)
    referable = d.binary_label.eq(1)
    report.append({
        'seed': seed,
        'auto_clear_n': int(clear.sum()),
        'missed_referable_n': int((clear & referable).sum()),
        'missed_grade2_n': int((clear & g2).sum()),
        'grade2_total_n': int(g2.sum()),
        'risk_pct': 100*(clear & referable).sum()/max(int(clear.sum()),1),
        'grade2_miss_pct': 100*(clear & g2).sum()/max(int(g2.sum()),1),
    })
report = pd.DataFrame(report).sort_values('seed')
source = res12[(res12.dataset=='IDRiD') & (res12.analysis=='source_LTT')].set_index('seed')
for row in report.itertuples(index=False):
    ref = source.loc[row.seed]
    assert int(row.auto_clear_n) == int(ref.auto_clear_n)
    assert int(row.missed_referable_n) == int(ref.missed_referable_n)
    assert np.isclose(row.risk_pct, float(ref.risk_pct), atol=1e-5)

print('Grade-2 source-selected failure audit\n', report.round(3).to_string(index=False))
print('\nGrade-2 share of false-clear decisions across five evaluations:',
    f'{report.missed_grade2_n.sum()}/{report.missed_referable_n.sum()} = '
    f'{100*report.missed_grade2_n.sum()/report.missed_referable_n.sum():.2f}%')
print('Repeated evaluations use the SAME 507 IDRiD images, not independent patients.')


In [ ]:
# CELL 6 — Experiment 15: retrospective 5% risk–coverage feasibility
required = {'dataset','seed','source_LTT_coverage_pct','source_LTT_risk_pct',
            'oracle_max_coverage_pct','oracle_risk_pct','oracle_auto_clear_n',
            'oracle_missed_ref_n','oracle_grade2_missed_n'}
assert required.issubset(res15.columns),required-set(res15.columns)
assert len(res15)==10
assert set(res15.seed)==SEEDS
assert not res15.duplicated(['dataset','seed']).any()
assert (res15.oracle_risk_pct <= 5 + 1e-8).all()
assert (res15.oracle_grade2_missed_n <= res15.oracle_missed_ref_n).all()
assert np.allclose(100*res15.oracle_missed_ref_n/res15.oracle_auto_clear_n,
                   res15.oracle_risk_pct, atol=1e-5)

# Reconcile source-selected numbers against the saved SR-GBLS results
source = res12[res12.analysis.eq('source_LTT')].set_index(['dataset','seed'])
for row in res15.itertuples(index=False):
    ref = source.loc[(row.dataset,row.seed)]
    assert np.isclose(row.source_LTT_coverage_pct,ref.coverage_pct,atol=1e-5)
    assert np.isclose(row.source_LTT_risk_pct,ref.risk_pct,atol=1e-5)

res15 = res15.copy()
res15['coverage_loss_pp'] = (res15.source_LTT_coverage_pct -
                              res15.oracle_max_coverage_pct)
feasibility = (res15.groupby('dataset')
    .agg(n_seeds=('seed','nunique'),
         source_coverage_mean=('source_LTT_coverage_pct','mean'),
         source_coverage_sd=('source_LTT_coverage_pct','std'),
         source_risk_mean=('source_LTT_risk_pct','mean'),
         source_risk_sd=('source_LTT_risk_pct','std'),
         retrospective_coverage_mean=('oracle_max_coverage_pct','mean'),
         retrospective_coverage_sd=('oracle_max_coverage_pct','std'),
         retrospective_risk_mean=('oracle_risk_pct','mean'),
         retrospective_risk_sd=('oracle_risk_pct','std'),
         coverage_loss_mean_pp=('coverage_loss_pp','mean'))
    .round(3))
assert feasibility.n_seeds.eq(5).all()
print('Recorded retrospective feasibility results\n',feasibility.to_string())
print('\nIMPORTANT: Retrospective cutoffs use external reference labels.')
print('They are NOT source-selected, externally calibrated, or LTT-certified policies.')
